In [ ]:
import polars as pl
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pyarrow as pa
import time
from collections import Counter
import gc
import chess
from sklearn.linear_model import SGDClassifier
import numpy as np
import os
import statistics
from collections import defaultdict
import re

In [ ]:
# A 3 perces mérkőzések körében hány alkalommal hajtott végre világos 'en passant' ütést olyan játszmákban, amelyek indiai típusú megnyitással kezdődtek? (Az ECO kód első betűje a megnyitás típusát jelöli)
pl.Config.set_streaming_chunk_size(1000000)

def solve_en_passant_blazing_fast():

    lf_valid_games = (
        pl.scan_parquet("games.parquet")
        .filter(
            (pl.col("timecontrol").str.starts_with("180+")) &
            (pl.col("eco").str.starts_with("E"))
        )
        .select("game_id")
    )

    lf_moves_filtered = (
        pl.scan_parquet("moves.parquet")
        .join(lf_valid_games, on="game_id", how="inner")
    )

    lf_white = (
        lf_moves_filtered
        .filter(pl.col("color") == "white")
        .filter(pl.col("move").str.contains("x") & pl.col("move").str.contains("6"))
        .filter(pl.col("move").str.contains(r"^[a-h]x[a-h]6[+#]?$"))
        .with_columns([
            (pl.col("move_no") - 1).alias("prev_move_no"),
            pl.col("move").str.extract(r"^[a-h]x([a-h])6").alias("target_file")
        ])
    )

    lf_black = (
        lf_moves_filtered
        .filter(pl.col("color") == "black")
        .filter(pl.col("move").str.contains("5"))
        .filter(pl.col("move").str.contains(r"^[a-h]5[+#]?$"))
        .with_columns([
            pl.col("move").str.extract(r"^([a-h])5").alias("push_file")
        ])
    )

    result = (
        lf_white.join(
            lf_black,
            left_on=["game_id", "prev_move_no", "target_file"],
            right_on=["game_id", "move_no", "push_file"],
            how="inner"
        )
        .select(pl.len().alias("count"))
        .collect(streaming=True)
    )

    count = result["count"][0]
    print(f"\n MEGOLDÁS: Világos {count} alkalommal ütött 'en passant'!")
    
    return count

solve_en_passant_blazing_fast()

1. Lekérdezési terv (Query Plan) felépítése...
2. Számítás indítása a processzormagokon... (Kérlek várj pár másodpercet)


C:\Users\gabor\AppData\Local\Temp\ipykernel_18048\2209705900.py:62: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)



🏆 MEGOLDÁS: Világos 15654 alkalommal ütött 'en passant'!


15654

In [ ]:
# Melyik játékos(ok) ad(tak) legtöbbször sáncolással mattot? (holtverseny esetén abc szerint első 10)
def solve_castling_mates_safe():

    df_moves_mates = (
        pl.scan_parquet("moves.parquet")
        .filter(pl.col("move").is_in(["O-O#", "O-O-O#"]))
        .select(["game_id", "color"])
        .collect()  
    )
    
    mate_count = len(df_moves_mates)
    print(f"-> Sikeresen találtunk {mate_count} db sáncolásos mattot. (Memória használat minimális)")
    
    if mate_count == 0:
        print("Nem található ilyen lépés az adatbázisban.")
        return
        
    valid_games = df_moves_mates["game_id"].to_list()
    
    
    df_games_filtered = (
        pl.scan_parquet("games.parquet")
        .filter(pl.col("game_id").is_in(valid_games))
        .select(["game_id", "white", "black"])
        .collect()  
    )
    

    result = (
        df_moves_mates.join(df_games_filtered, on="game_id", how="inner")
        .with_columns(
            pl.when(pl.col("color") == "white")
            .then(pl.col("white"))
            .otherwise(pl.col("black"))
            .alias("winner_name")
        )
        .group_by("winner_name")
        .agg(pl.len().alias("mate_count"))
        .sort(["mate_count", "winner_name"], descending=[True, False])
        .limit(10)
    )
    
    print("\n A legtöbbször sáncolással mattot adó játékosok (Top 10):")
    with pl.Config(tbl_rows=10):
        print(result)

solve_castling_mates_safe()

1. Ritka sáncolásos mattok keresése a 23GB-os lépéstáblában...
-> Sikeresen találtunk 683 db sáncolásos mattot. (Memória használat minimális)
2. A játékosnevek kinyerése a games táblából...
3. Eredmények összekapcsolása és aggregálása a memóriában...

🏆 A legtöbbször sáncolással mattot adó játékosok (Top 10):
shape: (10, 2)
┌──────────────────┬────────────┐
│ winner_name      ┆ mate_count │
│ ---              ┆ ---        │
│ str              ┆ u32        │
╞══════════════════╪════════════╡
│ HuberVilla       ┆ 6          │
│ Fly-Low-Hit-Hard ┆ 4          │
│ RumijaBarCG21    ┆ 4          │
│ VadimirUlaov     ┆ 4          │
│ chessyriy        ┆ 4          │
│ AntonBerg        ┆ 3          │
│ BlackGambit1     ┆ 3          │
│ FakriID          ┆ 3          │
│ NEWBYRONDUPIES   ┆ 3          │
│ vova1968         ┆ 3          │
└──────────────────┴────────────┘


In [ ]:
# Az adatbázisban lévő partik közül hány olyan játszma volt, ami közép-európai idő szerint potenciálisan elhúzódhatott (volna) egyik évről a másikra a játszma time kontrollja és a megtett lépések alapján csak „standard” alapállású partikat figyelembe véve? --> eredmény: minden olyan szilveszterre, ami beleesik az adatfelvétel time range-ébe 1db szám

pl.Config.set_streaming_chunk_size(1000000)

def solve_new_year_crossover_bulletproof():

    lf_nye_games = (
        pl.scan_parquet("games.parquet")
        .filter(pl.col("variant") == "Standard")
        .filter(pl.col("timecontrol").str.contains(r"^\d+\+\d+$"))
        .filter(pl.col("utcdate").str.contains(r"\.12\.30$|\.12\.31$"))

        .with_columns(
            (pl.col("utcdate") + " " + pl.col("utctime")).alias("utc_datetime_str")
        )
        .with_columns(
            pl.col("utc_datetime_str").str.strptime(pl.Datetime, "%Y.%m.%d %H:%M:%S", strict=False).alias("start_utc")
        )
        .drop_nulls("start_utc")
        .with_columns(
            (pl.col("start_utc") + pl.duration(hours=1)).alias("start_cet")
        )
        .filter(
            (pl.col("start_cet").dt.month() == 12) & 
            (pl.col("start_cet").dt.day() == 31)
        )
        .with_columns([
            pl.col("timecontrol").str.split("+").list.get(0).cast(pl.Int64).alias("base_sec"),
            pl.col("timecontrol").str.split("+").list.get(1).cast(pl.Int64).alias("inc_sec")
        ])
        .select(["game_id", "start_cet", "base_sec", "inc_sec"])
    )

    result = (
        pl.scan_parquet("moves.parquet")
        .select("game_id") 
        .join(lf_nye_games, on="game_id", how="inner")
        .group_by(["game_id", "start_cet", "base_sec", "inc_sec"])
        .agg(pl.len().alias("total_plies"))
        
        .with_columns(
            ((pl.col("base_sec") * 2) + (pl.col("total_plies") * pl.col("inc_sec"))).alias("max_duration_sec")
        )
        .with_columns(
            (pl.col("start_cet") + pl.duration(seconds=pl.col("max_duration_sec"))).alias("end_cet_potential")
        )
        .filter(
            pl.col("end_cet_potential").dt.year() > pl.col("start_cet").dt.year()
        )
        .group_by(pl.col("start_cet").dt.year().alias("szilveszter_eve"))
        .agg(pl.len().alias("atnyulo_partik_szama"))
        .sort("szilveszter_eve")
        .collect(streaming=True) 
    )

    print("\n EREDMÉNY: Szilveszteri meccsek, amik potenciálisan elhúzódtak a következő évbe:")
    with pl.Config(tbl_rows=-1):
        print(result)

solve_new_year_crossover_bulletproof()

1. Szilveszteri partik villámgyors szűrése a 'gyorsítósáv' technikával...
2. Lépésszámok összekapcsolása és aggregálás (Streaming módban)...


C:\Users\gabor\AppData\Local\Temp\ipykernel_20284\2691086573.py:70: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True) # Itt indul el az adatfolyam a processzorokon



🏆 EREDMÉNY: Szilveszteri meccsek, amik potenciálisan elhúzódtak a következő évbe:
shape: (3, 2)
┌─────────────────┬──────────────────────┐
│ szilveszter_eve ┆ atnyulo_partik_szama │
│ ---             ┆ ---                  │
│ i32             ┆ u32                  │
╞═════════════════╪══════════════════════╡
│ 2023            ┆ 42                   │
│ 2024            ┆ 4                    │
│ 2025            ┆ 67                   │
└─────────────────┴──────────────────────┘


In [ ]:
# Hogyan alakult a megfigyelt években 04.21 és 05.18. között (CET szerint) a vezércsellel kezdett partik százalékos aránya az összes játszmához képest csak „standard” alapállású partikat figyelembe véve? --> eredmény: minden év megjelölt időszakára, aminél szerepel legalább 1db játszma az adatbázisban 1db arányszám (vezércsellel kezdett partik száma / összes parti) kiegészítés: vezércsellel kezdett partinak minősül minden olyan játszma, ahol az első lépések 1. d4-d5; 2. c4 volt

pl.Config.set_streaming_chunk_size(1000000)

def solve_queens_gambit_bulletproof():
    
    df_spring_games = (
        pl.scan_parquet("games.parquet")
        .filter(pl.col("variant") == "Standard")
        .filter(pl.col("utcdate").str.slice(5, 2).is_in(["04", "05"]))
        .with_columns(
            (pl.col("utcdate") + " " + pl.col("utctime"))
            .str.strptime(pl.Datetime, "%Y.%m.%d %H:%M:%S", strict=False)
            .alias("utc_dt")
        )
        .drop_nulls("utc_dt")
        .with_columns((pl.col("utc_dt") + pl.duration(hours=2)).alias("cet_dt"))
        .with_columns([
            pl.col("cet_dt").dt.year().alias("cet_year"),
            pl.col("cet_dt").dt.month().alias("cet_month"),
            pl.col("cet_dt").dt.day().alias("cet_day")
        ])
        .filter(
            ((pl.col("cet_month") == 4) & (pl.col("cet_day") >= 21)) |
            ((pl.col("cet_month") == 5) & (pl.col("cet_day") <= 18))
        )
        .select(["game_id", "cet_year"])
        .collect() 
    )

    df_totals = df_spring_games.group_by("cet_year").agg(pl.len().alias("total_games"))

    if len(df_spring_games) == 0:
        return

    df_qg_moves = (
        pl.scan_parquet("moves.parquet")
        
        .filter(pl.col("move_no") <= 2)
        
        .filter(
            ((pl.col("move_no") == 1) & (pl.col("color") == "white") & (pl.col("move") == "d4")) |
            ((pl.col("move_no") == 1) & (pl.col("color") == "black") & (pl.col("move") == "d5")) |
            ((pl.col("move_no") == 2) & (pl.col("color") == "white") & (pl.col("move") == "c4"))
        )
        .select("game_id")
        
        .join(df_spring_games.lazy(), on="game_id", how="inner")
        
        .collect(streaming=True)
    )

    df_qg_counts = (
        df_qg_moves
        .group_by(["cet_year", "game_id"])
        .agg(pl.len().alias("match_count"))
        .filter(pl.col("match_count") == 3)
        .group_by("cet_year")
        .agg(pl.len().alias("qg_games"))
    )

    result = (
        df_totals.join(df_qg_counts, on="cet_year", how="left")
        .fill_null(0)
        .with_columns([
            (((pl.col("qg_games") / pl.col("total_games")) * 100).round(2).cast(pl.String) + "%").alias("Keresett_Arany")
        ])
        .sort("cet_year")
        .select(["cet_year", "total_games", "qg_games", "Keresett_Arany"])
    )

    print("EREDMÉNY: Vezércsel aránya évenként (04.21 - 05.18):")
    with pl.Config(tbl_rows=-1):
        print(result)

solve_queens_gambit_bulletproof()

1. Tavaszi (04.21 - 05.18) Standard partik lekérése...
2. A nyitólépések letapogatása a 23GB-os fájlban (Self-Join NÉLKÜL)...


C:\Users\gabor\AppData\Local\Temp\ipykernel_19596\684857970.py:63: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)


3. Eredmények számolása...

🏆 EREDMÉNY: Vezércsel aránya évenként (04.21 - 05.18):
shape: (2, 4)
┌──────────┬─────────────┬──────────┬────────────────┐
│ cet_year ┆ total_games ┆ qg_games ┆ Keresett_Arany │
│ ---      ┆ ---         ┆ ---      ┆ ---            │
│ i32      ┆ u32         ┆ u32      ┆ str            │
╞══════════╪═════════════╪══════════╪════════════════╡
│ 2024     ┆ 1070352     ┆ 37485    ┆ 3.5%           │
│ 2025     ┆ 2190611     ┆ 71980    ┆ 3.29%          │
└──────────┴─────────────┴──────────┴────────────────┘


In [ ]:
# Hány játszma végződött az 50 lépéses szabály miatt döntetlennel 2026.03.15 és 2026.10.14. között csak „standard” alapállású partikat figyelembe véve? --> eredmény: egyetlen szám

pl.Config.set_streaming_chunk_size(1000000)

def solve_50_move_rule_draws():
    
    df_draws = (
        pl.scan_parquet("games.parquet")
        .filter(pl.col("variant") == "Standard")
        .filter(pl.col("result") == "1/2-1/2")
        .filter(
            (pl.col("utcdate") >= "2026.03.15") & 
            (pl.col("utcdate") <= "2026.10.14")
        )
        
        .select("game_id")
        .collect() 
    )
    
    draw_count = len(df_draws)
    
    if draw_count == 0:
        return 0
        
    
    result_df = (
        pl.scan_parquet("moves.parquet")
        .select(["game_id", "move_no", "color", "move"])
        
        .join(df_draws.lazy(), on="game_id", how="inner")
        
        .with_columns([
            ((pl.col("move_no") * 2) - pl.when(pl.col("color") == "white").then(1).otherwise(0)).alias("ply"),
            
            (pl.col("move").str.contains("x") | pl.col("move").str.contains(r"^[a-h]")).alias("is_reset")
        ])
        
        .group_by("game_id")
        .agg([
            pl.col("ply").max().alias("max_ply"), 
            pl.col("ply").filter(pl.col("is_reset")).max().fill_null(0).alias("last_reset_ply") 
        ])
        
        .filter((pl.col("max_ply") - pl.col("last_reset_ply")) >= 100)
        
        .select(pl.len().alias("50_move_rule_draws"))
        .collect(streaming=True)
    )
    
    final_count = result_df["50_move_rule_draws"][0]
    print(f"\n EREDMÉNY: {final_count} db játszma végződött az 50 lépéses szabály miatt döntetlennel.")
    
    return final_count

solve_50_move_rule_draws()

1. Döntetlen partik és időablak villámgyors szűrése a games táblán...
-> Az időszakban lejátszott döntetlenek száma összesen: 27756
2. A 23GB-os tábla streamelése és a szabály matematikai ellenőrzése...


C:\Users\gabor\AppData\Local\Temp\ipykernel_19596\1170142806.py:64: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)



🏆 EREDMÉNY: 54 db játszma végződött az 50 lépéses szabály miatt döntetlennel.


54

In [ ]:
# Melyik játékosé, melyik időszakban és hány partin keresztül tartott a leghosszabb nyeretlenség széria csak „standard” alapállású partikat figyelembe véve? --> eredmény formátum: játékosnév, időszak (tól-ig), streak szám (ha több játékosnak azonos streak száma volt, akkor az nyert, akinek a játékosneve leghamarabb követi a „Lili” becenevet a magyar abc szabályai szerint) Kiegészítés: leghosszabb alatt a lejátszott játszmák darabszám és nem a naptári időszakot értjük. Folyamatban lévő streak is versenybe szállhat

def solve_winless_streak_hybrid_state_machine():
    
    pf = pq.ParquetFile("games.parquet")
    cols = ["variant", "white", "black", "result", "utcdate", "utctime"]
    

    player_states = {}
    
    
    for batch_idx, batch in enumerate(pf.iter_batches(batch_size=500000, columns=cols)):
        df = pl.from_arrow(batch)
        df = df.filter(pl.col("variant") == "Standard")
        
        if len(df) == 0:
            continue
            
        df = df.with_columns(
            (pl.col("utcdate") + " " + pl.col("utctime")).alias("dt")
        )
        
        df_white = df.select([
            pl.col("white").alias("player"), pl.col("dt"),
            (pl.col("result") == "1-0").cast(pl.Int8).alias("is_win")
        ])
        
        df_black = df.select([
            pl.col("black").alias("player"), pl.col("dt"),
            (pl.col("result") == "0-1").cast(pl.Int8).alias("is_win")
        ])
        
        df_events = pl.concat([df_white, df_black]).sort("dt")
        
        players = df_events["player"].to_list()
        dts = df_events["dt"].to_list()
        is_wins = df_events["is_win"].to_list()
        
        for p, dt, w in zip(players, dts, is_wins):
            state = player_states.get(p)
            
            if state is None:
                if w == 0:
                    player_states[p] = [1, dt, 1, dt, dt]
                else:
                    player_states[p] = [0, "", 0, "", ""]
            else:
                if w == 0:
                    if state[0] == 0:
                        state[1] = dt 
                    state[0] += 1
                    
                    if state[0] > state[2]:
                        state[2] = state[0]
                        state[3] = state[1]
                        state[4] = dt
                else:
                    state[0] = 0
                    
        del df, df_white, df_black, df_events, players, dts, is_wins

    
    global_max = 0
    for state in player_states.values():
        if state[2] > global_max:
            global_max = state[2]
            
    candidates = []
    for p, state in player_states.items():
        if state[2] == global_max:
            candidates.append({
                "player": p, "streak": state[2], "start": state[3], "end": state[4]
            })
            
    valid_candidates = [c for c in candidates if c["player"] > "Lili"]
    if valid_candidates:
        winner = sorted(valid_candidates, key=lambda x: x["player"])[0]
    else:
        winner = sorted(candidates, key=lambda x: x["player"])[0]
        
    print(" VÉGEREDMÉNY (Játékosnév, időszak, streak szám):")
    print(f"{winner['player']}, {winner['start']} - {winner['end']}, {winner['streak']}")

solve_winless_streak_hybrid_state_machine()

1. Adatfolyam megnyitása és Hibrid Állapotgép indítása...
2. Blokkok feldolgozása (Ez pár másodperc lesz, és 0 RAM/0 Lemez igényű!)...

3. Világrekorder megkeresése...
🏆 VÉGEREDMÉNY (Játékosnév, időszak, streak szám):
Varga-91, 2024.12.27 12:33:24 - 2024.12.27 14:03:41, 167


In [ ]:
# Melyik játékosé, melyik időszakban és hány partin keresztül tartott a leghosszabb döntetlen széria csak „standard” alapállású partikat figyelembe véve? --> eredmény: játékosnév, időszak (tól-ig), streak szám (ha több játékosnak azonos streak száma volt, akkor az nyert, akinek a széria utoló partijában a legmagasabb élőpontszáma volt) Kiegészítés: leghosszabb alatt a lejátszott játszmák darabszám és nem a naptári időszakot értjük. Folyamatban lévő streak is versenybe szállhat

def solve_longest_draw_streak():
    
    pf = pq.ParquetFile("games.parquet")
    
    cols = ["variant", "white", "black", "result", "utcdate", "utctime", "whiteelo", "blackelo"]
    
    player_states = {}
    
    print("2. Blokkok feldolgozása folyamatban (Ez a RAM-barát szakasz)...\n")
    
    for batch_idx, batch in enumerate(pf.iter_batches(batch_size=500000, columns=cols)):
        
        df = pl.from_arrow(batch)
        df = df.filter(pl.col("variant") == "Standard")
        
        if len(df) == 0:
            continue
            
        df = df.with_columns(
            (pl.col("utcdate") + " " + pl.col("utctime")).alias("dt")
        )
        
        df = df.with_columns([
            pl.col("whiteelo").cast(pl.Int32, strict=False).fill_null(0),
            pl.col("blackelo").cast(pl.Int32, strict=False).fill_null(0)
        ])
        
        df_white = df.select([
            pl.col("white").alias("player"), pl.col("dt"),
            (pl.col("result") == "1/2-1/2").cast(pl.Int8).alias("is_draw"),
            pl.col("whiteelo").alias("elo")
        ])
        
        df_black = df.select([
            pl.col("black").alias("player"), pl.col("dt"),
            (pl.col("result") == "1/2-1/2").cast(pl.Int8).alias("is_draw"),
            pl.col("blackelo").alias("elo")
        ])
        
        df_events = pl.concat([df_white, df_black]).sort("dt")
        
        players = df_events["player"].to_list()
        dts = df_events["dt"].to_list()
        is_draws = df_events["is_draw"].to_list()
        elos = df_events["elo"].to_list()
        
        for p, dt, is_draw, elo in zip(players, dts, is_draws, elos):
            state = player_states.get(p)
            
            if state is None:
                if is_draw == 1:
                    player_states[p] = [1, dt, 1, dt, dt, elo]
                else:
                    player_states[p] = [0, "", 0, "", "", 0]
            else:
                if is_draw == 1:
                    if state[0] == 0:
                        state[1] = dt
                    state[0] += 1
                    
                    if state[0] > state[2]:
                        state[2] = state[0] 
                        state[3] = state[1] 
                        state[4] = dt       
                        state[5] = elo      
                    
                    elif state[0] == state[2]:
                        if elo > state[5]:
                            state[3] = state[1]
                            state[4] = dt
                            state[5] = elo
                else:
                    state[0] = 0
                    
        del df, df_white, df_black, df_events, players, dts, is_draws, elos

    
    global_max = 0
    for state in player_states.values():
        if state[2] > global_max:
            global_max = state[2]
            
    candidates = []
    for p, state in player_states.items():
        if state[2] == global_max:
            candidates.append({
                "player": p, 
                "streak": state[2], 
                "start": state[3], 
                "end": state[4],
                "elo": state[5] 
            })
            
    winner = sorted(candidates, key=lambda x: x["elo"], reverse=True)[0]
        
    print(" VÉGEREDMÉNY (Játékosnév, időszak, streak szám):")
    print(f"{winner['player']}, {winner['start']} - {winner['end']}, {winner['streak']}")
    print(f"(A nyertes Élő-pontszáma a széria végén: {winner['elo']})")

solve_longest_draw_streak()

1. Adatfolyam megnyitása és Állapotgép indítása (Döntetlenek keresése)...
2. Blokkok feldolgozása folyamatban (Ez a RAM-barát szakasz)...

 -> Feldolgozva: 59500 ezer sor...

3. Világrekorder és a holtverseny (Élő-pontszám) kiértékelése...
🏆 VÉGEREDMÉNY (Játékosnév, időszak, streak szám):
chessvideworld, 2023.10.26 10:31:53 - 2023.10.26 10:51:39, 10
(A nyertes Élő-pontszáma a széria végén: 2104)


In [ ]:
# Mely napokon (datumokon) történt legalább egy olyan játszma, ahol az eredetileg az a2 mezőn álló világos gyalog eljutott a g8 mezőre és ott átváltozott? (elso 10 datum)
def solve_a2_to_g8_two_pass():
    pf = pq.ParquetFile("moves.parquet")

    candidates = set()
    start_time = time.time()
    
    for batch_idx, batch in enumerate(pf.iter_batches(batch_size=1000000, columns=["game_id", "color", "move"])):
        w_mask = pc.equal(batch.column("color"), "white")
        w_gids = batch.column("game_id").filter(w_mask)
        w_moves = batch.column("move").filter(w_mask)
        
        fxg8_mask = pc.starts_with(w_moves, "fxg8=")
        matched = w_gids.filter(fxg8_mask)
        
        if len(matched) > 0:
            candidates.update(matched.to_pylist())
            
        if batch_idx % 10 == 0: 
            rows = (batch_idx + 1) * 1000000
            speed = rows / (time.time() - start_time)
            
    
    if not candidates:
        return
        
    
    candidate_array = pa.array(list(candidates))
    
    matches = {gid: 0 for gid in candidates}
    bits = {"axb3": 1, "bxc4": 2, "cxd5": 4, "dxe6": 8, "exf7": 16, "fxg8=": 32}
    
    start_time = time.time()
    
    for batch_idx, batch in enumerate(pf.iter_batches(batch_size=1000000, columns=["game_id", "color", "move"])):
        in_mask = pc.is_in(batch.column("game_id"), value_set=candidate_array)
        valid_batch = batch.filter(in_mask)
        
        if valid_batch.num_rows > 0:
            w_mask = pc.equal(valid_batch.column("color"), "white")
            w_batch = valid_batch.filter(w_mask)
            
            if w_batch.num_rows > 0:
                gids = w_batch.column("game_id").to_pylist()
                moves = w_batch.column("move").to_pylist()
                
                for gid, move in zip(gids, moves):
                    for target, bit in bits.items():
                        if move.startswith(target):
                            matches[gid] |= bit
                            
        if batch_idx % 10 == 0:
            rows = (batch_idx + 1) * 1000000
            speed = rows / (time.time() - start_time)
            print(f" -> Ellenőrizve: {rows:,} sor | Sebesség: {speed:,.0f} sor/mp...", end="\r")

    winners = [gid for gid, bit in matches.items() if bit == 63]
    print(f"\n\n-> 2. Fázis kész! A mind a 6 ütést tartalmazó (a2->g8) játszmák száma: {len(winners)}")
    
    if not winners:
        print("Volt fxg8= lépés az adatbázisban, de egyik sem az a2 gyaloggal történt.")
        return

    print("\n3. LÉPÉS: Dátumok lekérdezése (Nincs 'Standard' szűrés!)...")
    
    lf_games = (
        pl.scan_parquet("games.parquet")
        .filter(pl.col("game_id").is_in(winners))
        .select("utcdate")
        .collect()
    )
    
    dates = lf_games["utcdate"].unique().sort().head(10).to_list()
    
    print("\n VÉGEREDMÉNY (Az első 10 dátum):")
    for i, date in enumerate(dates, 1):
        print(f"{i}. {date}")

solve_a2_to_g8_two_pass()

1. FÁZIS (Szűkítés): Csak az iszonyatosan ritka 'fxg8=' lépést keressük...
 -> Pásztázva: 4,061,000,000 sor | Sebesség: 9,766,126 sor/mp | Talált gyanús meccs: 161817

-> 1. Fázis kész! Összesen 16182 db gyanús játszmát találtunk. (RAM használat: ~0 MB)

2. FÁZIS (Ellenőrzés): A többi 5 ütés vizsgálata KIZÁRÓLAG a gyanús meccseknél...
 -> Ellenőrizve: 4,061,000,000 sor | Sebesség: 11,412,100 sor/mp...

-> 2. Fázis kész! A mind a 6 ütést tartalmazó (a2->g8) játszmák száma: 0
Volt fxg8= lépés az adatbázisban, de egyik sem az a2 gyaloggal történt.


In [ ]:
# Gyalogos átváltás: Hányszor nem királynőre váltották a gyalogot a beéréskor, és melyik a három legnépszerűbb bábu, amire átváltottak gyalogost királynő helyett (a váltások számát is meg kell adni)? (csak azokat az eseteket kell figyelembe venni amikor egyértelmű hogy mire lett átváltva a gyalogos)

def solve_underpromotions_ultra_fast():
    
    pf = pq.ParquetFile("moves.parquet")
    
    promo_counts = Counter()
    
    start_time = time.time()
    
    for batch_idx, batch in enumerate(pf.iter_batches(batch_size=2000000, columns=["move"])):
        moves = batch.column("move")
        
        eq_mask = pc.match_substring(moves, pattern="=")
        promo_batch = moves.filter(eq_mask)
        
        if len(promo_batch) > 0:
            valid_moves = promo_batch.to_pylist()
            
            for move in valid_moves:
                try:
                    eq_idx = move.index('=')
                    
                    if eq_idx + 1 < len(move):
                        piece = move[eq_idx + 1]
                        
                        if piece.isupper() and piece.isalpha() and piece != 'Q':
                            promo_counts[piece] += 1
                except ValueError:
                    continue
                    
        if batch_idx % 10 == 0:
            rows = (batch_idx + 1) * 2000000
            speed = rows / (time.time() - start_time)

    
    if not promo_counts:
        print("Nem találtunk a feltételeknek megfelelő átváltozást.")
        return
        
    total_underpromotions = sum(promo_counts.values())
    top_3 = promo_counts.most_common(3)
    
    piece_names = {
        "N": "Huszár (N)",
        "R": "Bástya (R)",
        "B": "Futó (B)",
        "K": "Király (K - Bár ez szabálytalan, ha benne van az adatban, látni fogjuk!)"
    }
    
    print(f"\n VÉGEREDMÉNY:")
    print(f"Összes 'nem Királynőre' történő átváltás száma: {total_underpromotions} db")
    print("\nA 3 legnépszerűbb bábu (Underpromotion):")
    
    for i, (piece, count) in enumerate(top_3, 1):
        name = piece_names.get(piece, f"Ismeretlen ({piece})")
        print(f"{i}. {name}: {count} alkalommal")

solve_underpromotions_ultra_fast()

1. LÉPÉS: A 23 GB-os moves.parquet pásztázása (Underpromotion keresés)...
 -> Pásztázva: 4,062,000,000 sor | Sebesség: 15,089,592 sor/mp...

2. LÉPÉS: Eredmények kiértékelése...

🏆 VÉGEREDMÉNY:
Összes 'nem Királynőre' történő átváltás száma: 207847 db

A 3 legnépszerűbb bábu (Underpromotion):
1. Bástya (R): 120443 alkalommal
2. Huszár (N): 63976 alkalommal
3. Futó (B): 23428 alkalommal


In [ ]:
#- Meccsenként inkább azok nyernek nagyobb arányban akik több vagy kevesebb időt használnak fel a másik játékoshoz képest?
GAMES_FILE    = "games.parquet"
MOVES_FILE    = "moves.parquet"

N_PARTITIONS  = 1000  

def hms_to_secs(col: str) -> pl.Expr:
    parts = pl.col(col).str.replace(r"\..*", "").str.split(":")
    return (
        pl.when(parts.list.len() == 2)
        .then(
            parts.list.get(0).cast(pl.Int64, strict=False) * 60
            + parts.list.get(1).cast(pl.Int64, strict=False)
        )
        .when(parts.list.len() == 3)
        .then(
            parts.list.get(0).cast(pl.Int64, strict=False) * 3600
            + parts.list.get(1).cast(pl.Int64, strict=False) * 60
            + parts.list.get(2).cast(pl.Int64, strict=False)
        )
        .otherwise(0)
    )

more_time_wins = 0
less_time_wins = 0


global_start_time = time.time()

for partition in range(N_PARTITIONS):
    part_start_time = time.time()
    print(f"Partition {partition + 1}/{N_PARTITIONS} ...", end=" ", flush=True)

    games_part = (
        pl.scan_parquet(GAMES_FILE, low_memory=True)
        .select(["game_id", "result", "whitestart", "blackstart"])
        .filter(pl.col("result").is_in(["1-0", "0-1"]))
        .filter(pl.col("game_id").hash(seed=42) % N_PARTITIONS == partition)
        .with_columns([
            hms_to_secs("whitestart").alias("ws"),
            hms_to_secs("blackstart").alias("bs"),
        ])
        .drop(["whitestart", "blackstart"])
        .collect()
    )

    if games_part.is_empty():
        print("empty, skipped.")
        continue

    moves_part = (
        pl.scan_parquet(MOVES_FILE, low_memory=True)
        .select(["game_id", "color", "move_no", "clock"])
        .filter(pl.col("game_id").hash(seed=42) % N_PARTITIONS == partition)
        .with_columns(
            pl.col("move_no").cast(pl.Int32)
        )
        .group_by(["game_id", "color"])
        .agg(
            pl.col("clock").gather(
                pl.col("move_no").arg_max().reshape([1])
            ).first().alias("last_clk_str")
        )
        .with_columns(
            hms_to_secs("last_clk_str").alias("last_clk")
        )
        .drop("last_clk_str")
        .collect()
    )

    white_clk = (
        moves_part
        .filter(pl.col("color") == "white")
        .select(["game_id", pl.col("last_clk").alias("wlc")])
    )
    black_clk = (
        moves_part
        .filter(pl.col("color") == "black")
        .select(["game_id", pl.col("last_clk").alias("blc")])
    )

    combined = (
        games_part
        .join(white_clk, on="game_id", how="inner")
        .join(black_clk,  on="game_id", how="inner")
        .with_columns([
            (pl.col("ws") - pl.col("wlc")).clip(lower_bound=0).alias("white_used"),
            (pl.col("bs") - pl.col("blc")).clip(lower_bound=0).alias("black_used"),
        ])
        .filter(pl.col("white_used") != pl.col("black_used"))
        .with_columns([
            pl.when(pl.col("white_used") > pl.col("black_used"))
              .then(pl.lit("white"))
              .otherwise(pl.lit("black"))
              .alias("more_time_color"),
            pl.when(pl.col("result") == "1-0")
              .then(pl.lit("white"))
              .otherwise(pl.lit("black"))
              .alias("winner_color"),
        ])
        .with_columns([
            (pl.col("more_time_color") == pl.col("winner_color")).alias("more_won")
        ])
    )

    part_more = combined["more_won"].sum()
    part_less = len(combined) - part_more
    more_time_wins += part_more
    less_time_wins += part_less

    elapsed = time.time() - part_start_time
    print(f"games={len(games_part):,}  analysed={len(combined):,}  more_wins={part_more:,}  ({elapsed:.1f} sec)")

    del games_part, moves_part, white_clk, black_clk, combined
    gc.collect()

total     = more_time_wins + less_time_wins
rate_more = more_time_wins / total if total else 0.0
rate_less = less_time_wins / total if total else 0.0

total_time = (time.time() - global_start_time) / 60

print()
print("=" * 60)
print("Q13 – Does the player using MORE time win more often?")
print("=" * 60)
print(f"  Decisive games analysed  : {total:>12,}")
print(f"  More-time player wins    : {more_time_wins:>12,}   ({rate_more:.4%})")
print(f"  Less-time player wins    : {less_time_wins:>12,}   ({rate_less:.4%})")
print()
if rate_more > rate_less:
    print(f"  → A TÖBB időt használó játékos nyer nagyobb arányban.")
    print(f"     Különbség: {rate_more - rate_less:+.4%}")
else:
    print(f"  → A KEVESEBB időt használó játékos nyer nagyobb arányban.")
    print(f"     Különbség: {rate_less - rate_more:+.4%}")
print(f"\n[Teljes futási idő: {total_time:.1f} perc]")
print("=" * 60)

Processing 1000 partitions (Memory Safe Mode)
Partition 1/1000 ... 

games=57,487  analysed=54,548  more_wins=16,350  (22.5 sec)
Partition 2/1000 ... games=57,046  analysed=54,139  more_wins=16,357  (26.3 sec)
Partition 3/1000 ... games=57,364  analysed=54,320  more_wins=16,405  (21.4 sec)
Partition 4/1000 ... games=57,364  analysed=54,477  more_wins=16,536  (20.0 sec)
Partition 5/1000 ... games=57,432  analysed=54,446  more_wins=16,487  (16.7 sec)
Partition 6/1000 ... games=57,728  analysed=54,825  more_wins=16,434  (19.6 sec)
Partition 7/1000 ... games=57,604  analysed=54,638  more_wins=16,602  (20.0 sec)
Partition 8/1000 ... games=57,272  analysed=54,279  more_wins=16,369  (25.2 sec)
Partition 9/1000 ... games=57,219  analysed=54,355  more_wins=16,421  (24.4 sec)
Partition 10/1000 ... games=57,016  analysed=54,123  more_wins=16,357  (27.0 sec)
Partition 11/1000 ... games=57,506  analysed=54,566  more_wins=16,276  (23.9 sec)
Partition 12/1000 ... games=57,583  analysed=54,604  more_wins=16,425  (24.9 sec)
Partition 13/1000 ... games=57,767  analysed=5

In [ ]:
#Feladás: Ki az, aki a legtöbbször adta fel a partit, hányan vannak akik nem adták fel soha, és hányan tartoznak a mediánba a feladások számát tekintve?
GAMES_FILE = "games.parquet"
MOVES_FILE = "moves.parquet"
F_MATES = "tmp_mates.parquet"

N_PARTITIONS = 100

def solve_resignations_flawless_phase3():

    global_start_t = time.time()
    if os.path.exists(F_MATES):
        try: os.remove(F_MATES)
        except: pass

    try:
        (
            pl.scan_parquet(MOVES_FILE)
            .filter(pl.col("move").str.ends_with("#"))
            .select("game_id")
            .sink_parquet(F_MATES)
        )


        resignation_counter = Counter()

        for partition in range(N_PARTITIONS):
            part_start_t = time.time()

            games_part = (
                pl.scan_parquet(GAMES_FILE, low_memory=True)
                .select(["game_id", "white", "black", "result", "termination"])
                .filter(pl.col("result").is_in(["1-0", "0-1"]))
                .filter(pl.col("termination") == "Normal")
                .filter(pl.col("game_id").hash(seed=42) % N_PARTITIONS == partition)
                .collect()
            )

            if games_part.is_empty():
                continue

            mates_part = (
                pl.scan_parquet(F_MATES, low_memory=True)
                .filter(pl.col("game_id").hash(seed=42) % N_PARTITIONS == partition)
                .collect()
            )

            resigns = games_part.join(mates_part, on="game_id", how="anti")

            w_resigns = resigns.filter(pl.col("result") == "0-1")["white"].to_list()
            b_resigns = resigns.filter(pl.col("result") == "1-0")["black"].to_list()

            resignation_counter.update(w_resigns)
            resignation_counter.update(b_resigns)

            if (partition + 1) % 10 == 0:
                print(f"  ... {partition + 1}/{N_PARTITIONS} partíció feldolgozva.", end="\r")

            del games_part, mates_part, resigns, w_resigns, b_resigns
            gc.collect()

        
        unique_players = set()
        pf_games = pq.ParquetFile(GAMES_FILE)

        for batch in pf_games.iter_batches(batch_size=500_000, columns=["white", "black"]):
            w_list = batch.column("white").to_pylist()
            b_list = batch.column("black").to_pylist()
            
            unique_players.update(w_list)
            unique_players.update(b_list)
            
            del w_list, b_list, batch
            gc.collect()

        total_players = len(unique_players)
        
        del unique_players
        gc.collect()
        
        print(f"   ✓ {total_players:,} egyedi játékos sikeresen megszámolva.")

        if resignation_counter:
            max_player, max_count = resignation_counter.most_common(1)[0]
        else:
            max_player, max_count = "Senki", 0

        zero_count = total_players - len(resignation_counter)

        all_counts = list(resignation_counter.values()) + [0] * zero_count
        
        median_val = statistics.median(all_counts)
        median_players = sum(1 for c in all_counts if c == median_val)

        total_time = (time.time() - global_start_t) / 60

        print("\n" + "=" * 70)
        print(f"  Összes vizsgált egyedi játékos: {total_players:,}")
        print("-" * 70)
        print(f"  1. A LEGTÖBBSZÖR adta fel    : {max_player} ({max_count:,} alkalommal)")
        print(f"  2. SOHA nem adták fel        : {zero_count:,} játékos")
        print(f"  3. Feladások MEDIÁNJA        : {median_val}")
        print(f"  -> Mediánba eső játékosok    : {median_players:,} fő")
        print("-" * 70)
        print(f"  Teljes futási idő            : {total_time:.1f} perc")
        print("=" * 70)

    except Exception as e:
        print(f"\n✗ Hiba történt: {e}")
        import traceback
        traceback.print_exc()
    finally:
        if os.path.exists(F_MATES):
            try: os.remove(F_MATES)
            except: pass

solve_resignations_flawless_phase3()

FELADÁSOK STATISZTIKÁJA (Szigorú 100 Partíció - Fagyásmentes 3. fázis)
1. Mattok kigyűjtése a 23 GB-os fájlból (Zéró RAM mód)...
   ✓ Mattok lementve az átmeneti fájlba.

2. Játékok feldolgozása és Anti-Join 100 partícióban...
  ... 100/100 partíció feldolgozva.

3. Összes játékos megszámlálása (PyArrow kötegelt olvasással)...
   ✓ 853,511 egyedi játékos sikeresen megszámolva.

  Összes vizsgált egyedi játékos: 853,511
----------------------------------------------------------------------
  1. A LEGTÖBBSZÖR adta fel    : siddeep (13,564 alkalommal)
  2. SOHA nem adták fel        : 213,888 játékos
  3. Feladások MEDIÁNJA        : 2
  -> Mediánba eső játékosok    : 84,740 fő
----------------------------------------------------------------------
  Teljes futási idő            : 10.0 perc


In [ ]:
#- Melyik felhasználó(k) szenvedte(k) el a legtöbb vereséget időtúllépés miatt olyan játszmákban, ahol a mérkőzés kezdetén élt(ek) a 'Berserk' opcióval? (holtverseny esetén abc szerint első 10)
GAMES_FILE = "games.parquet"

def parse_timecontrol(tc_str):
    if not tc_str or tc_str == "-": return None
    try:
        return int(tc_str.split('+')[0])
    except:
        return None

def parse_clock(c_str):
    if not c_str or c_str == "-": return None
    try:
        parts = str(c_str).split('.')[0].split(':')
        if len(parts) == 2: return int(parts[0]) * 60 + int(parts[1])
        if len(parts) == 3: return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
    except:
        pass
    return None

def solve_berserk_time_forfeits():

    start_t = time.time()
    
    berserk_losers = Counter()

    try:
        pf = pq.ParquetFile(GAMES_FILE)
        g_col = "game_id" if "game_id" in pf.schema.names else "id"
        
        batch_count = 0
        rows_processed = 0
        
        columns_to_read = ["white", "black", "result", "termination", "timecontrol", "whitestart", "blackstart"]
        
        for batch in pf.iter_batches(batch_size=500_000, columns=columns_to_read):
            batch_count += 1
            rows_processed += batch.num_rows
            
            whites = batch.column("white").to_pylist()
            blacks = batch.column("black").to_pylist()
            results = batch.column("result").to_pylist()
            terms = batch.column("termination").to_pylist()
            tcs = batch.column("timecontrol").to_pylist()
            w_starts = batch.column("whitestart").to_pylist()
            b_starts = batch.column("blackstart").to_pylist()
            
            for w, b, res, term, tc, ws, bs in zip(whites, blacks, results, terms, tcs, w_starts, b_starts):
                if term == "Time forfeit" and res in ("1-0", "0-1") and tc != "-":
                    base_time_sec = parse_timecontrol(tc)
                    if base_time_sec is None: continue
                    
                    if res == "0-1":
                        ws_sec = parse_clock(ws)
                        if ws_sec is not None and ws_sec < base_time_sec:
                            if w: berserk_losers[w] += 1
                            
                    elif res == "1-0":
                        bs_sec = parse_clock(bs)
                        if bs_sec is not None and bs_sec < base_time_sec:
                            if b: berserk_losers[b] += 1
                            

            del whites, blacks, results, terms, tcs, w_starts, b_starts, batch
            gc.collect()
            
        if not berserk_losers:
            print("\n✗ Nem találtunk ilyen játékost az adatbázisban.")
            return
            
        max_losses = max(berserk_losers.values())
        
        top_players = [player for player, count in berserk_losers.items() if count == max_losses]
        
        top_players.sort(key=lambda x: x.lower())
        
        top_10_players = top_players[:10]

        print("\n" + "=" * 70)
        print(f"  A LEGTÖBB BERSERK-IDŐTÚLLÉPÉSES VERESÉGET SZENVEDTÉK:")
        print(f"  {max_losses} vereséggel")
        print("-" * 70)
        for i, player in enumerate(top_10_players, 1):
            print(f"  {i}. {player}")
            
        if len(top_players) > 10:
            print(f"  ... és még {len(top_players) - 10} játékos ugyanennyi vereséggel.")
        print("=" * 70)

    except Exception as e:
        print(f"\n✗ Végzetes hiba történt: {e}")

solve_berserk_time_forfeits()

BERSERK IDŐTÚLLÉPÉSEK (Villámgyors PyArrow Mód)
Mérkőzések letapogatása...

  ... 59,306,757 mérkőzés átnézve.

✓ Adatok feldolgozva (765.3 mp).

  A LEGTÖBB BERSERK-IDŐTÚLLÉPÉSES VERESÉGET SZENVEDTÉK:
  -> 5547 vereséggel
----------------------------------------------------------------------
  1. zelkovahi


In [ ]:
#- Hány alkalommal végződött döntetlennel a játszma március 20-án olyan esetekben, ahol a mérkőzést lezáró utolsó lépés egy gyalog vezérré történő átváltozása volt?

GAMES_FILE = "games.parquet"
MOVES_FILE = "moves.parquet"

def solve_march20_promotions_smart():
    start_t = time.time()

    try:
        
        target_ids = (
            pl.scan_parquet(GAMES_FILE)
            .filter(pl.col("result") == "1/2-1/2")
            .filter(pl.col("utcdate").str.ends_with(".03.20"))
            .select("game_id")
            .collect()
        ).get_column("game_id").to_list()


        if not target_ids:
            return


        df_filtered_moves = (
            pl.scan_parquet(MOVES_FILE)
            .filter(pl.col("game_id").is_in(target_ids))
            .collect() 
        )



        promotions = (
            df_filtered_moves
            .with_columns(
                pl.when(pl.col("color") == "white").then(0).otherwise(1).alias("color_val")
            )
            .sort(["game_id", "move_no", "color_val"])
            .group_by("game_id")
            .last()
            .filter(pl.col("move").str.contains("=Q"))
        )

        promo_count = len(promotions)

        print("\n" + "=" * 70)
        print(f"  EREDMÉNY: {promo_count} alkalommal végződött így a játszma!")
        print("=" * 70)

    except Exception as e:
        print(f"\n✗ Hiba történt: {e}")

solve_march20_promotions_smart()

MÁRCIUS 20. DÖNTETLEN + VEZÉRÁTVÁLTOZÁS (Okos 'is_in' Szűrés)
1. Fázis: Március 20-i döntetlenek kigyűjtése...
   ✓ 5,757 mérkőzés azonosítva a RAM-ban.

2. Fázis: A 23 GB-os fájl szűrése az is_in() varázslattal...
   ✓ Szűrés kész! Csak 617,521 lépést kellett a memóriába tölteni.

3. Fázis: Utolsó lépések elemzése a memóriában...

  EREDMÉNY: 72 alkalommal végződött így a játszma!
  Példák a meccsekre:
  -> Game ID: 0GlMbf04-1049 | Utolsó lépés: b1=Q
  -> Game ID: HPpqa6n4-865 | Utolsó lépés: c1=Q
  -> Game ID: 1zbgslCT-81 | Utolsó lépés: a1=Q
  -> Game ID: eRMJHcGO-305 | Utolsó lépés: f1=Q
  -> Game ID: ODPtPV3N-682 | Utolsó lépés: a1=Q

[Teljes futási idő: 22.5 másodperc]


In [ ]:
# Hány játszma végződött háromszori lépésismétlés miatt döntetlennel 2024.03.12. és 2024.11.19. között csak „standard” alapállású partikat figyelembe véve? --> eredmény: egyetlen szám
GAMES_FILE = "games.parquet"
MOVES_FILE = "moves.parquet"

N_PARTITIONS = 100

def solve_threefold_repetition_partitioned():
    
    global_start_t = time.time()
    repetition_count = 0
    total_games_checked = 0
    
    try:
        print(f"Szimuláció indítása {N_PARTITIONS} memóriavédett lépcsőben...\n", flush=True)

        for partition in range(N_PARTITIONS):
            part_start_t = time.time()

            games_part = (
                pl.scan_parquet(GAMES_FILE, low_memory=True)
                .select(["game_id", "result", "termination", "variant", "utcdate"])
                .filter(pl.col("result") == "1/2-1/2")
                .filter(pl.col("termination") == "Normal")
                .filter(pl.col("variant") == "Standard")
                .filter((pl.col("utcdate") >= "2024.03.12") & (pl.col("utcdate") <= "2024.11.19"))
                .filter(pl.col("game_id").hash(seed=42) % N_PARTITIONS == partition)
                .collect()
            )

            if games_part.is_empty():
                continue

            target_ids = games_part["game_id"].to_list()
            total_games_checked += len(target_ids)


            moves_part = (
                pl.scan_parquet(MOVES_FILE, low_memory=True)
                .filter(pl.col("game_id").hash(seed=42) % N_PARTITIONS == partition)
                .filter(pl.col("game_id").is_in(target_ids))
                .select(["game_id", "move_no", "color", "move"])
                .collect()
            )

            games_dict = (
                moves_part
                .with_columns(
                    pl.when(pl.col("color") == "white").then(pl.lit(0)).otherwise(pl.lit(1)).alias("c_val")
                )
                .sort(["game_id", "move_no", "c_val"])
                .group_by("game_id")
                .agg(pl.col("move").alias("moves_list"))
                .to_dicts()
            )

            for row in games_dict:
                board = chess.Board()
                try:
                    for move_san in row["moves_list"]:
                        if move_san:
                            board.push_san(move_san)
                    

                    if board.is_stalemate() or board.is_insufficient_material():
                        continue
                        
                    if board.can_claim_threefold_repetition() or board.is_repetition(3):
                        repetition_count += 1
                except Exception:
                    pass

            elapsed = time.time() - part_start_t

            del games_part, target_ids, moves_part, games_dict
            gc.collect()

        total_time = (time.time() - global_start_t) / 60

        print("\n\n" + "=" * 70)
        print(f"  VÉGEREDMÉNY:")
        print(f"  Összesen vizsgált potenciális döntetlen: {total_games_checked:,}")
        print("-" * 70)
        print(f"  HÁROMSZORI LÉPÉSISMÉTLÉSEK SZÁMA: {repetition_count:,}")
        print(f"  Teljes futási idő: {total_time:.1f} perc")
        print("=" * 70)

    except Exception as e:
        print(f"\n✗ Hiba történt: {e}")
        import traceback
        traceback.print_exc()

solve_threefold_repetition_partitioned()

HÁROMSZORI LÉPÉSISMÉTLÉS (Szigorú 100 Partíciós Mód)
Szimuláció indítása 100 memóriavédett lépcsőben...

  ... 100/100 partíció kész. (Eddig vizsgált meccsek: 338,882)

  VÉGEREDMÉNY:
  Összesen vizsgált potenciális döntetlen: 338,882
----------------------------------------------------------------------
  HÁROMSZORI LÉPÉSISMÉTLÉSEK SZÁMA: 166,401
  Teljes futási idő: 37.7 perc


In [ ]:
# logit

GAMES_FILE    = "games.parquet"
MOVES_FILE    = "moves.parquet"

N_PARTITIONS  = 5000  

def hms_to_secs(col: str) -> pl.Expr:
    parts = pl.col(col).str.replace(r"\..*", "").str.split(":")
    return (
        pl.when(parts.list.len() == 2)
        .then(
            parts.list.get(0).cast(pl.Int64, strict=False) * 60
            + parts.list.get(1).cast(pl.Int64, strict=False)
        )
        .when(parts.list.len() == 3)
        .then(
            parts.list.get(0).cast(pl.Int64, strict=False) * 3600
            + parts.list.get(1).cast(pl.Int64, strict=False) * 60
            + parts.list.get(2).cast(pl.Int64, strict=False)
        )
        .otherwise(0)
    )


global_start_time = time.time()

clf_m1 = SGDClassifier(loss='log_loss', penalty=None, random_state=42)
clf_m2 = SGDClassifier(loss='log_loss', penalty=None, random_state=42)
classes = np.array([0, 1])

for partition in range(N_PARTITIONS):
    part_start_time = time.time()
    print(f"Partition {partition + 1}/{N_PARTITIONS} ...", end=" ", flush=True)

    games_part = (
        pl.scan_parquet(GAMES_FILE, low_memory=True)
        .select(["game_id", "result", "whitestart", "blackstart"])
        .filter(pl.col("game_id").hash(seed=42) % N_PARTITIONS == partition)
        .with_columns([
            hms_to_secs("whitestart").alias("ws_sec"),
            hms_to_secs("blackstart").alias("bs_sec"),
        ])
        .drop(["whitestart", "blackstart"])
        .collect()
    )

    if games_part.is_empty():
        print("empty, skipped.")
        continue

    moves_part = (
        pl.scan_parquet(MOVES_FILE, low_memory=True)
        .select(["game_id", "color", "move", "clock"])
        .filter(pl.col("game_id").hash(seed=42) % N_PARTITIONS == partition)
        .with_columns([
            (pl.col("color") == "white").cast(pl.Int8).alias("is_white"),
            pl.col("move").str.contains("x").fill_null(False).cast(pl.Int8).alias("is_capture"),
            hms_to_secs("clock").alias("clock_sec")
        ])
        .drop(["color", "move", "clock"])
        .collect()
    )

    if moves_part.is_empty():
        del games_part, moves_part
        gc.collect()
        print("empty moves, skipped.")
        continue

    joined = (
        moves_part.join(games_part, on="game_id", how="inner")
        .with_columns([
            pl.when(pl.col("is_white") == 1)
              .then(pl.col("ws_sec"))
              .otherwise(pl.col("bs_sec"))
              .alias("start_sec")
        ])
        .with_columns([
            (pl.col("start_sec") - pl.col("clock_sec")).alias("elapsed_time")
        ])
        .filter(pl.col("elapsed_time").is_not_null())
        .with_columns([
            pl.when(pl.col("elapsed_time") < 0).then(0).otherwise(pl.col("elapsed_time"))
        ])
    )

    if joined.is_empty():
        del games_part, moves_part, joined
        gc.collect()
        print("no valid times, skipped.")
        continue

    X1 = joined[["elapsed_time", "is_white"]].to_numpy()
    y1 = joined["is_capture"].to_numpy()
    clf_m1.partial_fit(X1, y1, classes=classes)

    m2_chunk = (
        joined
        .group_by(["game_id", "is_white", "result", "start_sec"])
        .agg([
            pl.col("is_capture").sum().alias("captures"),
            pl.len().alias("moves"),
            pl.col("clock_sec").min().alias("final_clock")
        ])
        .with_columns([
            (pl.col("start_sec") - pl.col("final_clock")).alias("total_time_spent")
        ])
        .with_columns([
            pl.when(pl.col("total_time_spent") < 0).then(0).otherwise(pl.col("total_time_spent")).alias("total_time_spent")
        ])
        .with_columns([
            (pl.col("total_time_spent") / pl.col("moves")).alias("avg_time_per_move"),
            (
                ((pl.col("result") == "1-0") & (pl.col("is_white") == 1)) | 
                ((pl.col("result") == "0-1") & (pl.col("is_white") == 0))
            ).cast(pl.Int8).alias("is_win")
        ])
        .drop_nulls()
    )

    y2 = m2_chunk["is_win"].to_numpy()
    X2 = m2_chunk[["captures", "is_white", "avg_time_per_move"]].to_numpy()
    if len(y2) > 0:
        clf_m2.partial_fit(X2, y2, classes=classes)

    elapsed = time.time() - part_start_time
    print(f"games={len(games_part):,}  moves={len(joined):,}  ({elapsed:.1f} sec)")

    del games_part, moves_part, joined, X1, y1, m2_chunk, X2, y2
    gc.collect()


print(">>> VÉGSŐ REGRESSZIÓS PARAMÉTEREK <<<")


print(f"\n--- MODELL 1 (Lépés szintű - Ütés Valószínűsége) ---")
print(f"Függő változó : is_capture (1=ütött, 0=nem)")
print(f"Konstans (Intercept) : {clf_m1.intercept_[0]:.6f}")
print(f"elapsed_time együtth : {clf_m1.coef_[0][0]:.6f}")
print(f"is_white együttható  : {clf_m1.coef_[0][1]:.6f}")

print(f"\n--- MODELL 2 (Játékos szintű - Győzelem Valószínűsége) ---")
print(f"Függő változó : is_win (1=nyert, 0=nem)")
print(f"Konstans (Intercept) : {clf_m2.intercept_[0]:.6f}")
print(f"total_captures együt : {clf_m2.coef_[0][0]:.6f}")
print(f"is_white együttható  : {clf_m2.coef_[0][1]:.6f}")
print(f"avg_time_per_move    : {clf_m2.coef_[0][2]:.6f}")

total_time = (time.time() - global_start_time) / 60
print(f"\n[Teljes futási idő: {total_time:.1f} perc]")
print("=" * 80)

LOGIT REGRESSZIÓK (5000 Partíciós Túlélő Mód)
Partition 1/5000 ... games=11,829  moves=807,885  (18.4 sec)
Partition 2/5000 ... games=11,739  moves=799,741  (19.3 sec)
Partition 3/5000 ... games=11,757  moves=803,316  (20.4 sec)
Partition 4/5000 ... games=11,995  moves=819,905  (19.1 sec)
Partition 5/5000 ... games=11,841  moves=808,034  (23.2 sec)
Partition 6/5000 ... games=11,962  moves=821,640  (21.2 sec)
Partition 7/5000 ... 

In [ ]:
#A versenygyőztesek által megnyert játszmákban a mattadás pillanatában átlagosan hány világos vezér tartózkodott a táblán?

global_start_time = time.time()

winner_set = set(
    pl.scan_parquet("tournaments.parquet")
    .select("winner__name")
    .unique()
    .collect()
    .to_series()
    .to_list()
)
winner_pa_array = pa.array(list(winner_set))

valid_games = set()
pf_games = pq.ParquetFile("games.parquet")

for batch in pf_games.iter_batches(batch_size=1000000, columns=["game_id", "variant", "white", "black", "result"]):
    is_standard = pc.equal(batch["variant"], "Standard")
    white_valid = pc.and_(pc.equal(batch["result"], "1-0"), pc.is_in(batch["white"], value_set=winner_pa_array))
    black_valid = pc.and_(pc.equal(batch["result"], "0-1"), pc.is_in(batch["black"], value_set=winner_pa_array))
    final_mask = pc.and_(is_standard, pc.or_(white_valid, black_valid))
    
    filtered_batch = batch.filter(final_mask)
    if filtered_batch.num_rows > 0:
        valid_games.update(filtered_batch["game_id"].to_pylist())
        
    del batch, filtered_batch, final_mask
    gc.collect()


pf_moves = pq.ParquetFile("moves.parquet")
total_rows = pf_moves.metadata.num_rows
processed_rows = 0

valid_games_series = pl.Series("vg", list(valid_games))
checkmate_games = set()

for batch in pf_moves.iter_batches(batch_size=2_000_000, columns=["game_id", "move"]):
    df = pl.from_arrow(batch)
    
    df = df.filter(pl.col("game_id").is_in(valid_games_series))
    
    mates_df = df.filter(pl.col("move").str.strip_chars().str.ends_with("#"))
    
    if mates_df.height > 0:
        checkmate_games.update(mates_df["game_id"].to_list())

    processed_rows += batch.num_rows
    print(f"\r   ... Radar haladás: {processed_rows:>12,} / {total_rows:,} ({(processed_rows/total_rows)*100:.1f}%)", end="", flush=True)

    del df, mates_df
    gc.collect()

del valid_games, valid_games_series 
gc.collect()


processed_rows = 0
checkmate_series = pl.Series("cm", list(checkmate_games))

active_boards = {}

checkmate_games_count = 0
total_white_queens = 0

for batch in pf_moves.iter_batches(batch_size=1_000_000, columns=["game_id", "move"]):
    df = pl.from_arrow(batch)
    
    valid_moves_df = df.filter(pl.col("game_id").is_in(checkmate_series))
    
    if valid_moves_df.height > 0:
        gids = valid_moves_df["game_id"].to_list()
        moves = valid_moves_df["move"].to_list()
        
        for gid, move in zip(gids, moves):
            if gid not in active_boards:
                active_boards[gid] = chess.Board()
                
            try:
                active_boards[gid].push_san(move)
                
                if move.strip().endswith("#"):
                    queens = len(active_boards[gid].pieces(chess.QUEEN, chess.WHITE))
                    total_white_queens += queens
                    checkmate_games_count += 1
                    
                    del active_boards[gid]
                    
            except ValueError:

                active_boards.pop(gid, None)

    processed_rows += batch.num_rows

    del df, valid_moves_df
    gc.collect()


print("\n" + "=" * 80)
if checkmate_games_count > 0:
    avg_queens = total_white_queens / checkmate_games_count
    print(f"MATT PILLANATÁBAN A TÁBLÁN LÉVŐ FEHÉR VEZÉREK ÁTLAGOS SZÁMA: {avg_queens:.4f}")
    print(f"(Alapul szolgáló mattos győztes meccsek száma: {checkmate_games_count:,})")
else:
    print("Nem találtunk a feltételeknek megfelelő mattos mérkőzést.")

total_time = (time.time() - global_start_time) / 60
print(f"\nTeljes futási idő: {total_time:.1f} perc")
print("=" * 80)

SAKK ADATELEMZÉS (Dupla-Radaros, RAM-védett Architektúra)
1. Versenygyőztesek betöltése...
   ✓ 21032 győztes betöltve.

2. Győztesek meccseinek kigyűjtése...
   ✓ Érvényes meccsek száma: 12,501,245

3. Fázis: A Matt-Radar futtatása (Csak a '#' végződésű meccsek keresése)...


C:\Users\gabor\AppData\Local\Temp\ipykernel_14984\2010145531.py:62: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df = df.filter(pl.col("game_id").is_in(valid_games_series))


   ... Radar haladás: 4,061,091,189 / 4,061,091,189 (100.0%)
   ✓ Radar végzett! Mattal végződő meccsek száma: 3,732,829 (Megmentettük a RAM-ot!)

4. Fázis: Lépések beolvasása és AZONNALI lejátszása (Zéró RAM tárolás)...


C:\Users\gabor\AppData\Local\Temp\ipykernel_14984\2010145531.py:97: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  valid_moves_df = df.filter(pl.col("game_id").is_in(checkmate_series))


   ... Motor haladás: 4,061,091,189 / 4,061,091,189 (100.0%) | Memóriában lévő aktív táblák: 0
   ✓ Motor végzett az összes meccsel!

MATT PILLANATÁBAN A TÁBLÁN LÉVŐ FEHÉR VEZÉREK ÁTLAGOS SZÁMA: 0.6450
(Alapul szolgáló mattos győztes meccsek száma: 3,732,829)

Teljes futási idő: 329.4 perc


In [ ]:
# Hány olyan partit játszottak 2023.10.12. és 2024.02.19. között, amikor a vesztes félnek legalább 3 pontnyi minőség hátránya volt csak „standard” alapállású partikat figyelembe véve? --> eredmény: egyetlen szám. Kiegészítés: használjuk a szokásos pontszámítást a figurák értékéhez, miszerint minden gyalog 1-et, minden futó és huszár 3-at, minden bástya 5-öt és minden vezér 9-et ér
PIECE_VALUES = {
    "P": 1, "N": 3, "B": 3, "R": 5, "Q": 9, "K": 0
}

def starting_piece(square):
    file, rank = square[0], int(square[1])

    if rank == 2:
        return ("P", "white")
    if rank == 7:
        return ("P", "black")

    if rank == 1:
        return ({
            "a": "R", "b": "N", "c": "B", "d": "Q",
            "e": "K", "f": "B", "g": "N", "h": "R"
        }[file], "white")

    if rank == 8:
        return ({
            "a": "R", "b": "N", "c": "B", "d": "Q",
            "e": "K", "f": "B", "g": "N", "h": "R"
        }[file], "black")

    return (None, None)

def extract_capture(move):
    m = re.search(r"x([a-h][1-8])", move)
    return m.group(1) if m else None

def extract_dest(move):
    m = re.findall(r"([a-h][1-8])", move)
    return m[-1] if m else None   # last square = destination

def extract_origin_hint(move):
    # helps remove correct piece (file hint like "Nbd2")
    m = re.match(r"[KQRBN]?([a-h])?[1-8]?", move)
    return m.group(1) if m else None

def extract_promotion(move):
    m = re.search(r"=([QRBN])", move)
    return m.group(1) if m else None

def piece_from_move(move):
    return move[0] if move[:1] in "KQRBN" else "P"

def handle_castling(move, color, board):
    if move.startswith("O-O-O"):  # long castle
        if color == "white":
            board.pop("e1", None)
            board.pop("a1", None)
            board["c1"] = ("K", "white")
            board["d1"] = ("R", "white")
        else:
            board.pop("e8", None)
            board.pop("a8", None)
            board["c8"] = ("K", "black")
            board["d8"] = ("R", "black")

    elif move.startswith("O-O"):  # short castle
        if color == "white":
            board.pop("e1", None)
            board.pop("h1", None)
            board["g1"] = ("K", "white")
            board["f1"] = ("R", "white")
        else:
            board.pop("e8", None)
            board.pop("h8", None)
            board["g8"] = ("K", "black")
            board["f8"] = ("R", "black")

def remove_origin_piece(board, piece, color, dest, origin_hint):
    for sq, (p, c) in list(board.items()):
        if p == piece and c == color:
            if origin_hint and not sq.startswith(origin_hint):
                continue
            del board[sq]
            return

games = (
    pl.scan_parquet("games.parquet")
    .with_columns(pl.col("date").str.replace_all(r"\.", "").alias("d"))
    .filter(
        (pl.col("variant") == "Standard") &
        (pl.col("result").is_in(["1-0", "0-1"])) &
        (pl.col("d") >= "20231012") &
        (pl.col("d") <= "20240219")
    )
    .select(["game_id", "result"])
    .collect()
)

game_map = {r["game_id"]: r["result"] for r in games.iter_rows(named=True)}

results = {}

current_game = None
board = {}
white_score = 0
black_score = 0

lazy_moves = pl.scan_parquet("moves.parquet").select(["game_id", "move", "color"])

for batch in lazy_moves.collect_batches():

    for row in batch.iter_rows(named=True):

        game_id = row["game_id"]

        if game_id not in game_map:
            continue

        move = row["move"]
        color = row["color"]

        if game_id != current_game:
            if current_game is not None:
                results[current_game] = (white_score, black_score)

            current_game = game_id
            board = {}
            white_score = 0
            black_score = 0

        if move.startswith("O-O"):
            handle_castling(move, color, board)
            continue

        capture_sq = extract_capture(move)
        dest = extract_dest(move)
        piece = piece_from_move(move)
        promo = extract_promotion(move)

        origin_hint = extract_origin_hint(move)
        remove_origin_piece(board, piece, color, dest, origin_hint)

        if capture_sq:
            if capture_sq in board:
                captured_piece, _ = board[capture_sq]
            else:
                captured_piece, _ = starting_piece(capture_sq)

            val = PIECE_VALUES.get(captured_piece, 0)

            if promo:
                val += PIECE_VALUES[promo] - 1

            if color == "white":
                white_score += val
            else:
                black_score += val

            new_piece = promo if promo else piece
            board[capture_sq] = (new_piece, color)

        else:
            if dest:
                new_piece = promo if promo else piece
                board[dest] = (new_piece, color)

                if promo:
                    val = PIECE_VALUES[promo] - 1
                    if color == "white":
                        white_score += val
                    else:
                        black_score += val

if current_game is not None:
    results[current_game] = (white_score, black_score)

count = 0

for gid, (w, b) in results.items():
    result = game_map[gid]

    if result == "1-0" and (w - b) >= 3:
        count += 1
    elif result == "0-1" and (b - w) >= 3:
        count += 1

print("Partik száma:", count)

In [ ]:
# Bal oldali lóval ütéses meccsek: Mennyivel nagyobb arányban nyertek azok a  játékosok akik ütöttek a bal oldali lóval azokhoz képest, akik nem ütöttek bal oldali lóval a játék során (bal oldali ló alatt a fehérnek a B1-es, feketének a G8-as mezőről induló ló számít)?
FILES = "abcdefgh"

def knight_moves(square):
    f = FILES.index(square[0])
    r = int(square[1]) - 1

    deltas = [(1,2),(2,1),(2,-1),(1,-2),(-1,-2),(-2,-1),(-2,1),(-1,2)]
    res = []

    for df, dr in deltas:
        nf, nr = f + df, r + dr
        if 0 <= nf < 8 and 0 <= nr < 8:
            res.append(FILES[nf] + str(nr+1))
    return set(res)

KNIGHT_MAP = {f + str(r): knight_moves(f + str(r)) for f in FILES for r in range(1,9)}

def is_knight_move(move):
    return move.startswith("N")

def extract_dest(move):
    m = re.findall(r"([a-h][1-8])", move)
    return m[-1] if m else None

def extract_disambiguation(move):
    m = re.match(r"N([a-h]?[1-8]?)(?:x)?([a-h][1-8])", move)
    if not m:
        return ""
    return m.group(1)

def is_capture(move):
    return "x" in move

games = (
    pl.scan_parquet("games.parquet")
    .filter(pl.col("variant") == "Standard")
    .select(["game_id", "result"])
    .collect()
)

game_map = {r["game_id"]: r["result"] for r in games.iter_rows(named=True)}

results = []

current_game = None

white_knight_pos = None
black_knight_pos = None

white_captured = False
black_captured = False

lazy_moves = pl.scan_parquet("moves.parquet").select(["game_id", "move", "color"])

for batch in lazy_moves.collect_batches():

    for row in batch.iter_rows(named=True):

        gid = row["game_id"]

        if gid not in game_map:
            continue

        move = row["move"]
        color = row["color"]

        if gid != current_game:
            if current_game is not None:
                results.append((white_captured, black_captured, game_map[current_game]))

            current_game = gid

            white_knight_pos = "b1"
            black_knight_pos = "g8"

            white_captured = False
            black_captured = False

        if not is_knight_move(move):
            continue

        dest = extract_dest(move)
        if not dest:
            continue

        disamb = extract_disambiguation(move)

        if color == "white":

            if dest in KNIGHT_MAP[white_knight_pos]:

                if disamb:
                    file_ok = white_knight_pos[0] in disamb
                    rank_ok = white_knight_pos[1] in disamb
                    if not (file_ok or rank_ok):
                        continue

                white_knight_pos = dest

                if is_capture(move):
                    white_captured = True

        else:

            if dest in KNIGHT_MAP[black_knight_pos]:

                if disamb:
                    file_ok = black_knight_pos[0] in disamb
                    rank_ok = black_knight_pos[1] in disamb
                    if not (file_ok or rank_ok):
                        continue

                black_knight_pos = dest

                if is_capture(move):
                    black_captured = True

if current_game is not None:
    results.append((white_captured, black_captured, game_map[current_game]))

captured_wins = 0
captured_total = 0

noncaptured_wins = 0
noncaptured_total = 0

for wc, bc, result in results:

    if wc:
        captured_total += 1
        if result == "1-0":
            captured_wins += 1
    else:
        noncaptured_total += 1
        if result == "1-0":
            noncaptured_wins += 1

    if bc:
        captured_total += 1
        if result == "0-1":
            captured_wins += 1
    else:
        noncaptured_total += 1
        if result == "0-1":
            noncaptured_wins += 1

winrate_captured = captured_wins / captured_total if captured_total else 0
winrate_not = noncaptured_wins / noncaptured_total if noncaptured_total else 0

print("Győzelmi arány különbség a bal lóval ütők és nem ütők között:", winrate_captured - winrate_not)

In [ ]:
# A 10 perces mérkőzések során hány esetben veszítette el világos a sáncoláshoz való jogát a játszma első három teljes lépésén (az első 6 fél-lépésen) belül?
games = (
    pl.scan_parquet("games.parquet")
    .filter(
        (pl.col("variant") == "Standard") &
        (pl.col("timecontrol") == "600+0")
    )
    .select("game_id")
    .collect()
)

game_ids = set(g["game_id"] for g in games.iter_rows(named=True))

lazy_moves = (
    pl.scan_parquet("moves.parquet")
    .filter(
        (pl.col("color") == "white") &
        (pl.col("move_no").is_in([2, 3]))
    )
    .select(["game_id", "move"])
)

games_with_king_move = set()

for batch in lazy_moves.collect_batches():

    for row in batch.iter_rows(named=True):

        gid = row["game_id"]

        if gid not in game_ids:
            continue

        move = row["move"]

        if move.startswith("K"):
            games_with_king_move.add(gid)

# =========================================================
# RESULT
# =========================================================

print("Number of games:", len(games_with_king_move))

In [ ]:
# Bástyák: Az összes fehér és fekete bástya lépése között mennyi a különbség megtett hosszban?
FILES = "abcdefgh"

def square_to_coords(s):
    return FILES.index(s[0]), int(s[1]) - 1

def rook_distance(a, b):
    f1, r1 = square_to_coords(a)
    f2, r2 = square_to_coords(b)
    return abs(f1 - f2) + abs(r1 - r2)

def extract_dest(move):
    m = re.findall(r"([a-h][1-8])", move)
    return m[-1] if m else None

def is_rook_move(move):
    return move.startswith("R")

def is_rook_promo(move):
    return "=R" in move

def is_capture(move):
    return "x" in move

def extract_disamb(move):
    m = re.match(r"R([a-h]?[1-8]?)(?:x)?([a-h][1-8])", move)
    if not m:
        return ""
    return m.group(1)

games = (
    pl.scan_parquet("games.parquet")
    .filter(pl.col("variant") == "Standard")
    .select(["game_id"])
    .collect()
)

game_ids = set(g["game_id"] for g in games.iter_rows(named=True))

white_total = 0
black_total = 0

current_game = None

rooks_white = set()
rooks_black = set()

lazy_moves = pl.scan_parquet("moves.parquet").select(["game_id", "move", "color"])

for batch in lazy_moves.collect_batches():

    for row in batch.iter_rows(named=True):

        gid = row["game_id"]
        if gid not in game_ids:
            continue

        move = row["move"]
        color = row["color"]

        if gid != current_game:
            current_game = gid
            rooks_white = {"a1", "h1"}
            rooks_black = {"a8", "h8"}

        if move.startswith("O-O-O"):
            if color == "white":
                rooks_white.discard("a1")
                rooks_white.add("d1")
                white_total += rook_distance("a1", "d1")
            else:
                rooks_black.discard("a8")
                rooks_black.add("d8")
                black_total += rook_distance("a8", "d8")
            continue

        if move.startswith("O-O"):
            if color == "white":
                rooks_white.discard("h1")
                rooks_white.add("f1")
                white_total += rook_distance("h1", "f1")
            else:
                rooks_black.discard("h8")
                rooks_black.add("f8")
                black_total += rook_distance("h8", "f8")
            continue

        if is_rook_promo(move):
            dest = extract_dest(move)
            if dest:
                if color == "white":
                    rooks_white.add(dest)
                else:
                    rooks_black.add(dest)
            continue

        if not is_rook_move(move):
            continue

        dest = extract_dest(move)
        if not dest:
            continue

        disamb = extract_disamb(move)

        rooks = rooks_white if color == "white" else rooks_black

        candidates = []

        for r in rooks:
            rf, rr = square_to_coords(r)
            df, dr = square_to_coords(dest)

            if rf == df or rr == dr:
                if disamb:
                    if not (r[0] in disamb or r[1] in disamb):
                        continue
                candidates.append(r)

        if not candidates:
            continue

        origin = candidates[0]

        dist = rook_distance(origin, dest)

        if color == "white":
            white_total += dist
        else:
            black_total += dist

        rooks.remove(origin)
        rooks.add(dest)

print("Lépéshosszkülönbség fehér és fekete bástyák között (fehér - fekete):", white_total - black_total)

In [ ]:
# Hány alkalommal ért véget egy játszma háromszori lépésismétléssel olyan játékosok részvételével, akiknek van egy olló emoji van a nevében?
TOURNAMENTS_FILE = "tournaments.parquet"
GAMES_FILE = "games.parquet"
MOVES_FILE = "moves.parquet"

N_PARTITIONS = 100


def solve_threefold_repetition_partitioned():

    repetition_count = 0
    total_games_checked = 0

    tournaments = pl.read_parquet(TOURNAMENTS_FILE)

    tournament_ids = (
        tournaments
        .filter(pl.col("winner__flair") == "objects.scissors")
        .select("id")
        .unique()
    )["id"].to_list()

    tournament_id_set = set(tournament_ids)

    games_lazy = pl.scan_parquet(GAMES_FILE)

    game_ids = (
        games_lazy
        .filter(pl.col("tournamentid").is_in(tournament_id_set))
        .select("game_id")
        .unique()
        .collect()
    )["game_id"].to_list()

    game_id_set = set(game_ids)

    moves_lazy = pl.scan_parquet(MOVES_FILE)

    for partition in range(N_PARTITIONS):

        partition_game_ids = [
            gid for gid in game_id_set
            if hash(gid) % N_PARTITIONS == partition
        ]

        if not partition_game_ids:
            continue

        total_games_checked += len(partition_game_ids)

        moves_part = (
            moves_lazy
            .filter(pl.col("game_id").is_in(partition_game_ids))
            .select(["game_id", "move_no", "color", "move"])
            .collect(streaming=True)
        )

        if moves_part.is_empty():
            continue

        games_dict = (
            moves_part
            .with_columns(
                pl.when(pl.col("color") == "white")
                .then(0)
                .otherwise(1)
                .alias("c_val")
            )
            .sort(["game_id", "move_no", "c_val"])
            .group_by("game_id")
            .agg(pl.col("move").alias("moves_list"))
            .to_dicts()
        )

        for row in games_dict:
            board = chess.Board()

            try:
                for move in row["moves_list"]:
                    if move:
                        board.push_san(move)

                if board.can_claim_threefold_repetition():
                    repetition_count += 1

            except Exception:
                pass

        del moves_part, games_dict
        gc.collect()

    print(f"Háromszoros repetícióval véget érő partik száma olyan játékosokkal, akiknek olló emoji van a nevében: {repetition_count:,}")

solve_threefold_repetition_partitioned()

In [ ]:
# Logit regresszió: Függő változó: Lépésekre generált dummy változó (1 ha azzal a lépéssel ütött, 0 ha nem) Magyarázó változók: meccs kezdete óta eltelt idő másodpercben, bábu színe (white=1) dummy. Mik ennek a modellnek a becsült paraméterei? & Logit regresszió: Függő változó: nyert-e an meccs, Magyarázó változók: A játékos ütéseinek száma, a játékos színe,  átlagosan mennyi időt használ fel a játékos egy lépésre. Mik ennek a modellnek a becsült paraméterei?
GAMES_FILE    = "games.parquet"
MOVES_FILE    = "moves.parquet"

N_PARTITIONS  = 3000  

def hms_to_secs(col: str) -> pl.Expr:
    parts = pl.col(col).str.replace(r"\..*", "").str.split(":")
    return (
        pl.when(parts.list.len() == 2)
        .then(
            parts.list.get(0).cast(pl.Int64, strict=False) * 60
            + parts.list.get(1).cast(pl.Int64, strict=False)
        )
        .when(parts.list.len() == 3)
        .then(
            parts.list.get(0).cast(pl.Int64, strict=False) * 3600
            + parts.list.get(1).cast(pl.Int64, strict=False) * 60
            + parts.list.get(2).cast(pl.Int64, strict=False)
        )
        .otherwise(0)
    )

clf_m1 = SGDClassifier(loss='log_loss', penalty=None, random_state=42)
clf_m2 = SGDClassifier(loss='log_loss', penalty=None, random_state=42)
classes = np.array([0, 1])

for partition in range(N_PARTITIONS):
    games_part = (
        pl.scan_parquet(GAMES_FILE, low_memory=True)
        .select(["game_id", "result", "whitestart", "blackstart"])
        .filter(pl.col("game_id").hash(seed=42) % N_PARTITIONS == partition)
        .with_columns([
            hms_to_secs("whitestart").alias("ws_sec"),
            hms_to_secs("blackstart").alias("bs_sec"),
        ])
        .drop(["whitestart", "blackstart"])
        .collect()
    )

    if games_part.is_empty():
        print("empty, skipped.")
        continue

    # ── STEP B: Lépések betöltése (Csak az aktuális partíció) ────────────────
    moves_part = (
        pl.scan_parquet(MOVES_FILE, low_memory=True)
        .select(["game_id", "color", "move", "clock"])
        .filter(pl.col("game_id").hash(seed=42) % N_PARTITIONS == partition)
        .with_columns([
            (pl.col("color") == "white").cast(pl.Int8).alias("is_white"),
            pl.col("move").str.contains("x").fill_null(False).cast(pl.Int8).alias("is_capture"),
            hms_to_secs("clock").alias("clock_sec")
        ])
        .drop(["color", "move", "clock"])
        .collect()
    )

    if moves_part.is_empty():
        del games_part, moves_part
        gc.collect()
        continue

    joined = (
        moves_part.join(games_part, on="game_id", how="inner")
        .with_columns([
            pl.when(pl.col("is_white") == 1)
              .then(pl.col("ws_sec"))
              .otherwise(pl.col("bs_sec"))
              .alias("start_sec")
        ])
        .with_columns([
            (pl.col("start_sec") - pl.col("clock_sec")).alias("elapsed_time")
        ])
        .filter(pl.col("elapsed_time").is_not_null())
        .with_columns([
            pl.when(pl.col("elapsed_time") < 0).then(0).otherwise(pl.col("elapsed_time"))
        ])
    )

    if joined.is_empty():
        del games_part, moves_part, joined
        gc.collect()
        continue

    X1 = joined[["elapsed_time", "is_white"]].to_numpy()
    y1 = joined["is_capture"].to_numpy()
    clf_m1.partial_fit(X1, y1, classes=classes)

    m2_chunk = (
        joined
        .group_by(["game_id", "is_white", "result", "start_sec"])
        .agg([
            pl.col("is_capture").sum().alias("captures"),
            pl.len().alias("moves"),
            pl.col("clock_sec").min().alias("final_clock")
        ])
        .with_columns([
            (pl.col("start_sec") - pl.col("final_clock")).alias("total_time_spent")
        ])
        .with_columns([
            pl.when(pl.col("total_time_spent") < 0).then(0).otherwise(pl.col("total_time_spent")).alias("total_time_spent")
        ])
        .with_columns([
            (pl.col("total_time_spent") / pl.col("moves")).alias("avg_time_per_move"),
            (
                ((pl.col("result") == "1-0") & (pl.col("is_white") == 1)) | 
                ((pl.col("result") == "0-1") & (pl.col("is_white") == 0))
            ).cast(pl.Int8).alias("is_win")
        ])
        .drop_nulls()
    )

    y2 = m2_chunk["is_win"].to_numpy()
    X2 = m2_chunk[["captures", "is_white", "avg_time_per_move"]].to_numpy()
    if len(y2) > 0:
        clf_m2.partial_fit(X2, y2, classes=classes)

    del games_part, moves_part, joined, X1, y1, m2_chunk, X2, y2
    gc.collect()

print(f"\n--- MODELL 1 (Lépés szintű - Ütés Valószínűsége) ---")
print(f"Függő változó : is_capture (1=ütött, 0=nem)")
print(f"Konstans (Intercept) : {clf_m1.intercept_[0]:.6f}")
print(f"elapsed_time együtth : {clf_m1.coef_[0][0]:.6f}")
print(f"is_white együttható  : {clf_m1.coef_[0][1]:.6f}")

print(f"\n--- MODELL 2 (Játékos szintű - Győzelem Valószínűsége) ---")
print(f"Függő változó : is_win (1=nyert, 0=nem)")
print(f"Konstans (Intercept) : {clf_m2.intercept_[0]:.6f}")
print(f"total_captures együt : {clf_m2.coef_[0][0]:.6f}")
print(f"is_white együttható  : {clf_m2.coef_[0][1]:.6f}")
print(f"avg_time_per_move    : {clf_m2.coef_[0][2]:.6f}")